# 05 — Predictive Modeling: Calibrated Churn, Discounted Value, and Expected Profit

**Owner:** M5 — Thư  
**Purpose:** run the professional, script-backed M5 pipeline.

This notebook is intentionally thin. The reusable production logic lives in `scripts/modeling.py`; this notebook is only for execution and review.

## What this notebook does

1. Validates the current repo structure and core input files.
2. Runs the M5 v3 end-to-end pipeline from `config/paths.yaml`.
3. Produces calibrated churn probabilities, two-part discounted 60-day value estimates, SHAP explainability outputs, seasonality audit files, and expected incremental profit rankings.
4. Keeps M4 unchanged; M5 consumes the delivered feature table as-is.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data').exists() and (PROJECT_ROOT.parent / 'Data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Config:', PROJECT_ROOT / 'config' / 'paths.yaml')

Project root: c:\Users\Thuww\DDM-Churn-Project
Config: c:\Users\Thuww\DDM-Churn-Project\config\paths.yaml


## 1. Validate core inputs

In [2]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(PROJECT_ROOT / 'config' / 'paths.yaml')
input_checks

{
  "feature_table_csv": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv",
    "exists": true,
    "size_bytes": 693945
  },
  "transaction_master_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet",
    "exists": true,
    "size_bytes": 24336697
  },
  "customer_base_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet",
    "exists": true,
    "size_bytes": 122150
  }
}


{'feature_table_csv': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv',
  'exists': True,
  'size_bytes': 693945},
 'transaction_master_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet',
  'exists': True,
  'size_bytes': 24336697},
 'customer_base_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet',
  'exists': True,
  'size_bytes': 122150}}

## 2. Run M5 pipeline

In [3]:
from scripts.modeling import run_m5_pipeline

summary = run_m5_pipeline(PROJECT_ROOT / 'config' / 'paths.yaml')
summary

[M5] Project root: C:\Users\Thuww\DDM-Churn-Project
[M5] Tuning classifier: Logistic Regression balanced
[M5]   Logistic Regression balanced: CV candidate 1/10
[M5]   Logistic Regression balanced: CV candidate 2/10
[M5]   Logistic Regression balanced: CV candidate 3/10


C:\Users\Thuww\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 10 is smaller than n_iter=40. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[M5]   Logistic Regression balanced: CV candidate 4/10
[M5]   Logistic Regression balanced: CV candidate 5/10
[M5]   Logistic Regression balanced: CV candidate 6/10
[M5]   Logistic Regression balanced: CV candidate 7/10
[M5]   Logistic Regression balanced: CV candidate 8/10
[M5]   Logistic Regression balanced: CV candidate 9/10
[M5]   Logistic Regression balanced: CV candidate 10/10
[M5] Tuning classifier: Random Forest balanced
[M5]   Random Forest balanced: CV candidate 1/40
[M5]   Random Forest balanced: CV candidate 2/40
[M5]   Random Forest balanced: CV candidate 3/40
[M5]   Random Forest balanced: CV candidate 4/40
[M5]   Random Forest balanced: CV candidate 5/40
[M5]   Random Forest balanced: CV candidate 6/40
[M5]   Random Forest balanced: CV candidate 7/40
[M5]   Random Forest balanced: CV candidate 8/40
[M5]   Random Forest balanced: CV candidate 9/40
[M5]   Random Forest balanced: CV candidate 10/40
[M5]   Random Forest balanced: CV candidate 11/40
[M5]   Random Forest balan

{'version': 'v3_discounted_two_part_value_shap_seasonality',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'XGBoost weighted',
 'calibration_method': 'sigmoid',
 'champion_threshold': 0.13,
 'test_PR_AUC_calibrated': 0.3652138015042468,
 'test_F2_score_calibrated': 0.4583333333333333,
 'test_brier_score_calibrated': 0.09506485629007785,
 'active_model': 'Active Random Forest__isotonic',
 'value_model': 'Random Forest Regressor',
 'annual_discount_rate': 0.08,
 'base_profitable_customers': 0,
 'shap_status_file': 'C:\\Users\\Thuww\\DDM-Churn-Project\\reports\\internal_briefs\\M5_shap_status.json',
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models'}

## 3. Review generated outputs

In [4]:
import pandas as pd

model_dir = PROJECT_ROOT / 'models'
metrics = pd.read_csv(model_dir / 'model_metrics.csv')
active_metrics = pd.read_csv(model_dir / 'active_model_metrics.csv')
value_metrics = pd.read_csv(model_dir / 'value_model_metrics.csv')
scenario_summary = pd.read_csv(model_dir / 'scenario_profit_summary.csv')
calibration_summary = pd.read_csv(model_dir / 'calibration_summary.csv')

display(metrics.head())
display(calibration_summary.head())
display(active_metrics.head())
display(value_metrics.head())
display(scenario_summary)


,model,tuned,features_used,best_cv_PR_AUC,best_params,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,...,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP,best_cv_PR_AUC_std
0,XGBoost weighted,True,numeric+categorical,0.368037,"{""model__colsample_bytree"": 0.75, ""model__lear...",0.377835,0.736790,0.515873,0.291045,0.639344,...,0.47,0.238,0.422645,0.12,0.302645,354,86,27,33,0.057377
1,Extra Trees balanced,True,numeric+categorical,0.355746,"{""model__max_depth"": 5, ""model__max_features"":...",0.373724,0.746817,0.486726,0.211538,0.721311,...,0.38,0.408,0.385642,0.12,0.265642,278,162,18,42,0.056799
2,Logistic Regression balanced,True,numeric+categorical,0.315523,"{""model__C"": 0.005}",0.366964,0.747265,0.483559,0.183150,0.819672,...,0.43,0.484,0.453930,0.12,0.333930,242,198,16,44,0.044709
3,Random Forest balanced,True,numeric+categorical,0.366619,"{""model__max_depth"": null, ""model__max_feature...",0.308891,0.726950,0.488851,0.168142,0.934426,...,0.09,0.682,0.184431,0.12,0.064431,152,288,7,53,0.066833
4,Dummy prior,False,none,NaN,{},0.122000,0.500000,0.409946,0.122000,1.000000,...,0.01,1.000,0.120747,0.12,0.000747,0,440,0,60,NaN


,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP,val_brier_gap_from_best,calibration_selection_eligible,selected_for_champion,selection_rule
0,XGBoost weighted,sigmoid,0.377835,0.73679,0.515873,0.291045,0.639344,0.096296,0.13,0.268,...,0.12,0.001438,353,87,27,33,0.009176,True,True,choose highest validation PR-AUC among calibra...
1,XGBoost weighted,raw_uncalibrated,0.377835,0.73679,0.515873,0.291045,0.639344,0.186154,0.47,0.268,...,0.12,0.302645,354,86,27,33,0.099033,False,False,choose highest validation PR-AUC among calibra...
2,XGBoost weighted,isotonic,0.369233,0.77305,0.517241,0.293233,0.639344,0.087120,0.09,0.266,...,0.12,0.006981,355,85,27,33,0.000000,True,False,choose highest validation PR-AUC among calibra...


,active_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_brier_score,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Random Forest,isotonic,0.984074,0.891328,0.980562,0.90982,1.0,0.060805,0.01,0.998,...,0.070896,0.01,1.0,0.900935,0.892,0.008935,0,54,0,446
1,Active Logistic Regression,isotonic,0.983525,0.889102,0.980562,0.90982,1.0,0.061413,0.01,0.998,...,0.075268,0.01,1.0,0.903431,0.892,0.011431,0,54,0,446
2,Active Random Forest,raw_uncalibrated,0.984240,0.877227,0.980562,0.90982,1.0,0.064937,0.46,0.998,...,0.071005,0.46,1.0,0.888579,0.892,-0.003421,0,54,0,446
3,Active Logistic Regression,raw_uncalibrated,0.983875,0.868991,0.980562,0.90982,1.0,0.068618,0.12,0.998,...,0.072582,0.12,1.0,0.891017,0.892,-0.000983,0,54,0,446
4,Active Random Forest,sigmoid,0.984240,0.877227,0.980562,0.90982,1.0,0.069956,0.60,0.998,...,0.078706,0.60,1.0,0.904456,0.892,0.012456,0,54,0,446


,value_model,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,Random Forest Regressor,log1p(discounted_future_revenue_60d | active),0.882681,0.656536,0.514891,139.381051,0.886086,0.642423,0.496243,143.766239
1,Ridge Regression,log1p(discounted_future_revenue_60d | active),0.918765,0.686662,0.474418,166.320471,0.957195,0.713307,0.412145,191.129702
2,XGBoost Regressor,log1p(discounted_future_revenue_60d | active),0.919749,0.674425,0.473292,143.681740,0.912005,0.661173,0.466342,149.714869


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.000000,0.000000,750,-3657.133459
1,base,named,0.25,0.05,5.0,0,0.000000,0.000000,750,-3556.528039
2,optimistic,named,0.30,0.08,3.0,4,0.001601,1.908624,750,-1878.533835


## 4. Handoff files for M6

Use `models/high_risk_customers_for_ab_test.csv` as the experiment population candidate file.

Use `models/profitable_treatment_candidates_base.csv` only as the profitability sanity-check file under base assumptions.

Important interpretation:
- `p_churn_calibrated` is the churn-risk score used for business formulas.
- `predicted_discounted_value_60d_if_active` is a discounted 60-day value estimate, not full lifetime CLV.
- `predicted_expected_discounted_value_60d` comes from the two-part model: `p_future_active × value_if_active`.
- SHAP files explain prediction drivers, not causal treatment effects.

## Giải thích chi tiết tất cả các chỉ số

### **PHẦN 1: Classification Metrics (Dự báo churn hay không)**

#### PR-AUC (Precision-Recall AUC)
- **Thang:** 0 → 1 (cao càng tốt)
- **Ý nghĩa:** Tỷ lệ giữa những dự báo churn đúng (precision) với tất cả những khách thực sự churn (recall)
- **Dùng khi nào:** Khi dữ liệu mất cân bằng (12% churn, 88% non-churn)
- **Ví dụ:** PR-AUC = 0.36 nghĩa là model tốt hơn baseline (0.12) gấp 3 lần

#### ROC-AUC
- **Thang:** 0 → 1 (cao càng tốt)
- **Ý nghĩa:** Tỷ lệ giữa True Positive Rate (nhận diện churn) vs False Positive Rate (cảnh báo sai)
- **Ưu điểm:** Không bị ảnh hưởng bởi sự mất cân bằng dữ liệu
- **Ví dụ:** ROC-AUC = 0.73 → model phân biệt churn/non-churn tốt

#### F2-Score
- **Công thức:** Trung bình có trọng số giữa Precision và Recall, ưu tiên Recall gấp 2 lần
- **Ý nghĩa:** Nhấn mạnh **nhận diện đầy đủ** churners (tránh bỏ sót)
- **Thang:** 0 → 1 (cao càng tốt)
- **Ví dụ:** F2 = 0.46 nghĩa là cân bằng tốt giữa dự báo chính xác và bắt được churn

#### Precision
- **Công thức:** (TP) / (TP + FP) = dự báo churn đúng / tất cả dự báo churn
- **Ý nghĩa:** Trong số những khách bạn nhận diện là churn, bao nhiêu % thực sự churn
- **Ví dụ:** Precision = 0.18 → chỉ 18% khách bạn nhận diện là churn là thực sự churn (68% sai lạc)

#### Recall
- **Công thức:** (TP) / (TP + FN) = dự báo churn đúng / tất cả churn thực tế
- **Ý nghĩa:** Bạn bắt được bao nhiêu % churners thực tế
- **Ví dụ:** Recall = 0.73 → bạn phát hiện được 73% khách hàng churn thực sự (bỏ sót 27%)

#### Brier Score
- **Công thức:** Trung bình bình phương sai lệch (predicted_prob - actual_label)²
- **Thang:** 0 → 1 (thấp càng tốt)
- **Ý nghĩa:** Độ chính xác của xác suất dự báo (không chỉ dự báo đúng/sai)
- **Ví dụ:** Brier = 0.093 → sai lệch xác suất trung bình ≈ 9.3 điểm phần trăm

#### Threshold
- **Ý nghĩa:** Ngưỡng xác suất để quyết định "churn hay không"
- **Mặc định:** 0.5 (xác suất churn ≥ 50% → dự báo churn)
- **Thực tế:** Có thể điều chỉnh để ưu tiên Precision hay Recall
- **Ví dụ:** Nếu threshold = 0.1, model sẽ cảnh báo churn khi xác suất ≥ 10% (nhạy hơn, nhưng sai nhiều hơn)

#### Predicted Positive Rate
- **Ý nghĩa:** % khách bạn dự báo là churn
- **Ví dụ:** 0.48 → dự báo 48% khách hàng sẽ churn

#### Mean Predicted Probability
- **Ý nghĩa:** Xác suất churn trung bình của tất cả khách
- **Ví dụ:** 0.45 → xác suất churn trung bình là 45%

#### Actual Positive Rate
- **Ý nghĩa:** % khách thực sự churn trong tập dữ liệu
- **Ví dụ:** 0.12 → 12% khách thực sự churn (không thay đổi)

#### Calibration Gap (Mean - Actual)
- **Công thức:** Mean Predicted Probability - Actual Positive Rate
- **Ý nghĩa:** Liệu xác suất dự báo có khớp với tỷ lệ thực tế không
- **Mục tiêu:** Gần 0 (không bias)
- **Ví dụ:** 0.33 → model dự báo quá cao (bias dương), nên hiệu chỉnh

#### Confusion Matrix (TN, FP, FN, TP)
```
                Predicted Churn=0    Predicted Churn=1
Actual Churn=0  TN (Đúng)           FP (Sai)
Actual Churn=1  FN (Sai)            TP (Đúng)
```
- **TN = True Negative:** Dự báo non-churn, thực tế non-churn ✓
- **FP = False Positive:** Dự báo churn, thực tế non-churn ✗ (cảnh báo sai lạc)
- **FN = False Negative:** Dự báo non-churn, thực tế churn ✗ (bỏ sót churners)
- **TP = True Positive:** Dự báo churn, thực tế churn ✓

---

### **PHẦN 2: Tuning Results Metrics**

#### mean_cv_PR_AUC
- **Ý nghĩa:** Điểm PR-AUC trung bình từ 5-fold cross-validation
- **Dùng để:** So sánh hyperparameter candidate nào tốt nhất (rank 1 = tốt nhất)

#### std_cv_PR_AUC
- **Ý nghĩa:** Độ lệch chuẩn của PR-AUC giữa 5 fold
- **Ý nghĩa thấp:** Model ổn định, không bị overfitting vào fold nào
- **Ý nghĩa cao:** Model kém ổn định, thay đổi khi dữ liệu khác

---

### **PHẦN 3: Value Model Metrics (Regression)**

#### RMSE_log (Root Mean Square Error trên log scale)
- **Công thức:** √(Trung bình (log(pred) - log(actual))²)
- **Thang:** 0 → ∞ (thấp càng tốt)
- **Ý nghĩa:** Sai lệch trung bình trong dự báo giá trị (trên thang log)
- **Ví dụ:** RMSE_log = 0.88 → sai lệch ~88% trên thang log

#### MAE_log (Mean Absolute Error trên log scale)
- **Công thức:** Trung bình |log(pred) - log(actual)|
- **Thang:** 0 → ∞ (thấp càng tốt)
- **Ý nghĩa:** Sai lệch trung bình tuyệt đối trên thang log

#### R2_log (Hệ số xác định trên log scale)
- **Thang:** -∞ → 1 (cao càng tốt, 1 = dự báo hoàn hảo)
- **Ý nghĩa:** % phương sai của giá trị được model giải thích
- **Ví dụ:** R2 = 0.51 → model giải thích được 51% biến động giá trị

#### MAE_revenue (Mean Absolute Error trên giá trị thực)
- **Thang:** 0 → ∞ (thấp càng tốt, đơn vị VND)
- **Ý nghĩa:** Sai lệch trung bình trong dự báo doanh thu (không dùng log)
- **Ví dụ:** MAE_revenue = 143k → trung bình sai lệch 143 nghìn VND

---

### **PHẦN 4: Scenario Profit Metrics**

#### expected_incremental_profit
- **Công thức:** (p_churn × save_rate × customer_value) - treatment_cost
- **Ý nghĩa:** Lợi nhuận kỳ vọng nếu bạn can thiệp (treatment) cho khách này
- **Ví dụ:** 
  - p_churn = 80%, customer_value = 10 triệu, save_rate = 5%, cost = 0.5 triệu
  - Profit = (0.8 × 0.05 × 10M) - 0.5M = 0.4M - 0.5M = -0.1M (lỗ!)

#### 3 kịch bản
1. **Conservative (Bảo thủ):**
   - Margin: 20%, Save rate: 3%, Cost: $5
   - Dùng khi: Không chắc chắn về effectiveness

2. **Base (Cơ sở):**
   - Margin: 25%, Save rate: 5%, Cost: $5
   - Dùng khi: Dữ liệu lịch sử cho thấy 5% save rate

3. **Optimistic (Lạc quan):**
   - Margin: 30%, Save rate: 8%, Cost: $5
   - Dùng khi: Chiến dịch tương tự trước đạt 8% save rate

---

### **PHẦN 5: Active Model Metrics (Phần 1 của 2-part model)**

#### p_future_active
- **Ý nghĩa:** Xác suất khách hàng sẽ **còn sống (active) trong 60 ngày tới**
- **Định nghĩa "active":** Có ít nhất 1 giao dịch trong 60 ngày sau cut-off day
- **Ví dụ:** p_future_active = 0.9 → 90% khách sẽ còn tiếp tục mua hàng

#### Tại sao cần phân 2 phần?
```
Doanh thu kỳ vọng 60 ngày = P(active) × E(doanh thu | active)
Không nên dùng: E(doanh thu) cho tất cả khách (sẽ bias)
```

---

### **PHẦN 6: Churn Predictions Output (high_risk_customers_for_ab_test.csv)**

| Cột | Ý nghĩa |
|-----|---------|
| `household_key` | ID khách hàng |
| `p_churn_calibrated` | Xác suất churn (0-1), hiệu chỉnh bằng isotonic |
| `churn_decile` | Xếp hạng theo rủi ro churn (1=cao nhất, 10=thấp nhất) |
| `p_future_active` | Xác suất còn sống trong 60 ngày |
| `predicted_discounted_value_60d_if_active` | Doanh thu dự báo nếu còn sống (VND, chiết khấu 8%) |
| `predicted_expected_discounted_value_60d` | Doanh thu kỳ vọng = p_active × value_if_active |
| `expected_incremental_profit` | Lợi nhuận nếu can thiệp (base scenario) |
| `profit_ranking_decile` | Xếp hạng theo lợi nhuận (1=cao nhất) |

---

### **🎯 TÓM TẮT: Cách lựa chọn khách hàng để can thiệp**

```
Ưu tiên = P(churn) cao + Value cao + Expected profit cao
Bỏ qua = P(churn) thấp hoặc Value thấp hoặc Profit âm
```